Modify the PCA class to support inverse_transform. Reconstruct MNIST digits from 10, 50, and 200 components. Print the reconstruction error (mean squared difference from the original) for each.

In [1]:
import numpy as np

class PCA:
    def __init__(self, n_components):
        self.n_components = n_components
        self.components = None
        self.mean = None
        self.eigenvalues = None
        self.explained_variance_ratio_ = None

    def fit(self, X):
        self.mean = np.mean(X, axis=0)

        X_centered = X - self.mean

        cov_matrix = np.cov(X_centered, rowvar=False)

        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

        # Sort eigenvalues from largest to smallest
        sorted_idx = np.argsort(eigenvalues)[::-1]

        eigenvalues = eigenvalues[sorted_idx]
        eigenvectors = eigenvectors[:, sorted_idx]

        # Keep only n_components
        self.components = eigenvectors[:, :self.n_components].T

        self.eigenvalues = eigenvalues[:self.n_components]

        total_var = np.sum(eigenvalues)

        self.explained_variance_ratio_ = (
            self.eigenvalues / total_var
        )

        return self

    def transform(self, X):
        X_centered = X - self.mean

        return X_centered @ self.components.T

    def inverse_transform(self, X_transformed):
        # Reconstruct the centered data
        X_centered = X_transformed @ self.components

        # Add the mean back
        X_reconstructed = X_centered + self.mean

        return X_reconstructed

    def fit_transform(self, X):
        self.fit(X)

        return self.transform(X)

In [3]:
import numpy as np
from sklearn.datasets import fetch_openml

# Load MNIST
mnist = fetch_openml("mnist_784", version=1, as_frame=False)

X = mnist.data
y = mnist.target

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (70000, 784)
y shape: (70000,)


In [4]:
X = X[:10000].astype(np.float64)

In [5]:
for n in [10, 50, 200]:

    pca = PCA(n_components=n)

    X_transformed = pca.fit_transform(X)

    X_reconstructed = pca.inverse_transform(X_transformed)

    mse = np.mean((X - X_reconstructed) ** 2)

    print(f"Components: {n}")
    print(f"Reconstruction error: {mse:.4f}")
    print()

Components: 10
Reconstruction error: 2212.4163

Components: 50
Reconstruction error: 751.4013

Components: 200
Reconstruction error: 141.0914

